# SELECT * FROM Korea — 유학생 데이터 탐색 (7 Steps EDA)

> 심사위원(Seon Ho Kim 교수님) **"7 Steps of Data Exploration"** 프레임워크 적용 (SSOT §5-2).
> 데이터: `data/raw/universities.csv` (대학 224개), `students_by_nationality.csv` (14,550행 · 유학생 197,163명).
> 목표는 체크리스트 나열이 아니라 **발표 인사이트 2개**를 뽑는 것: (1) 결측의 구조(MAR), (2) 파생변수(국적 다양성).

| Step | 내용 | 우리 데이터 |
|---|---|---|
| 1 | 변수 식별 | nominal / **ordinal**(체류자격) / continuous |
| 2 | 단변량 | 대학별 유학생 수 → 강한 우편향 |
| 3 | 이변량 | 설립구분×규모, 국적×체류자격 |
| 4 | **결측 처리** ⭐ | 기숙사비 결측이 소규모 대학에 집중 = MAR |
| 5 | 이상치 | 기숙사비 1원(인공) vs 120만원(자연) |
| 6 | 변환 | log 변환으로 우편향 완화 |
| 7 | **파생변수** ⭐ | 국적 다양성 지수(Shannon) |

In [ ]:
import pandas as pd, numpy as np
import matplotlib, matplotlib.pyplot as plt
from pathlib import Path

# 한글 폰트 (Windows: Malgun Gothic)
plt.rcParams['axes.unicode_minus'] = False
for _f in ['Malgun Gothic', 'AppleGothic', 'NanumGothic']:
    try:
        plt.rcParams['font.family'] = _f; break
    except Exception:
        pass

BASE = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RAW = BASE / 'data' / 'raw'
FIG = BASE / 'notebooks' / 'figures'; FIG.mkdir(parents=True, exist_ok=True)

u = pd.read_csv(RAW / 'universities.csv', encoding='utf-8-sig')
s = pd.read_csv(RAW / 'students_by_nationality.csv', encoding='utf-8-sig')
# 다중 캠퍼스(경동대_, 을지대_) 중복 → 큰 캠퍼스 대표값으로 정리
u = u.sort_values('total', ascending=False).groupby('univ_key', as_index=False).first()
print('universities:', u.shape, '| students:', s.shape, '| 유학생 합계:', int(s.headcount.sum()))

## Step 1 — 변수 식별

체류자격(`visa_status`)은 단순 명목형이 아니라 **순서형(ordinal)** 입니다: 어학연수 → 전문학사 → 학사 → 석사 → 박사. 이 순서를 살리면 '학위 단계가 올라갈수록 무엇이 달라지는가'를 볼 수 있습니다.

In [ ]:
nominal = ['region', 'establishment', 'school_type', 'nationality', 'gender', 'dorm_available']
ordinal = ['visa_status']  # 어학연수 < 전문학사 < 학사 < 석사 < 박사
continuous = ['total','degree_total','engineering','lang_qualified_ratio','dorm_capacity_rate','dorm_fee_min','headcount']
visa_order = ['대학부설 어학원 연수','외국어연수생','전문학사과정','학사과정','석사과정','박사과정','교환학생','(구)교환학생','학술연구기관 특정연구자']
print('nominal   :', nominal)
print('ordinal   :', ordinal, '→', visa_order[:6])
print('continuous:', continuous)

## Step 2 — 단변량 분석

대학별 유학생 총원은 **강하게 우편향(right-skewed)** 되어 있습니다. 소수의 초대형 대학과 다수의 소형 대학. 이 분포는 Step 6(로그 변환)의 근거가 됩니다.

In [ ]:
print(u['total'].describe().round(0))
print('왜도(skewness):', round(u['total'].skew(), 2))
print('상위5:', u.nlargest(5,'total')[['univ_name','total']].values.tolist())

fig, ax = plt.subplots(1, 2, figsize=(11,3.5))
ax[0].hist(u['total'], bins=30, color='#7f8c8d'); ax[0].set_title('유학생 총원 분포 (원본)'); ax[0].set_xlabel('total')
u['region'].value_counts().head(8).plot.bar(ax=ax[1], color='#2980b9'); ax[1].set_title('지역별 대학 수 (상위8)')
plt.tight_layout(); plt.show()

## Step 3 — 이변량 분석

설립구분·지역과 규모, 국적과 체류자격의 관계를 봅니다.

In [ ]:
# 국적 × 체류자격 교차표 (상위 국적)
top_nat = s.groupby('nationality').headcount.sum().nlargest(6).index
ct = (s[s.nationality.isin(top_nat)]
        .pivot_table(index='nationality', columns='visa_status', values='headcount', aggfunc='sum', fill_value=0))
deg_cols = [c for c in ['전문학사과정','학사과정','석사과정','박사과정'] if c in ct.columns]
print('상위 국적 × 학위과정 (명):'); print(ct[deg_cols])

fig, ax = plt.subplots(1, 2, figsize=(11,3.5))
u.boxplot(column='total', by='establishment', ax=ax[0]); ax[0].set_title('설립구분별 규모'); ax[0].set_xlabel(''); plt.suptitle('')
ax[1].scatter(u['degree_total'], u['lang_qualified_total'], s=12, alpha=.5, color='#8e44ad')
ax[1].set_xlabel('학위과정 유학생'); ax[1].set_ylabel('어학요건 충족 수'); ax[1].set_title('학위규모 vs 어학 준비도')
plt.tight_layout(); plt.show()

## Step 4 — 결측 처리 ⭐ (핵심 인사이트 1)

**가설:** 소규모 대학일수록 기숙사비(`dorm_fee_min`)를 공시하지 않는다. 사실이라면 결측은 무작위(MCAR)가 아니라 **규모에 의존하는 MAR**이고, 단순 삭제하면 *"기숙사비 분석이 대형 대학에 편향"* 되는 잘못된 결론이 나옵니다. (강의자료의 크리켓 예시와 같은 구조)

In [ ]:
miss = u['dorm_fee_min'].isna()
print('기숙사비 결측:', int(miss.sum()), '/', len(u))
print('결측군  규모 중앙값: %.0f' % u.loc[miss,'total'].median())
print('비결측군 규모 중앙값: %.0f' % u.loc[~miss,'total'].median())

u['size_q'] = pd.qcut(u['total'], 4, labels=['Q1\n(소)','Q2','Q3','Q4\n(대)'])
rate = u.groupby('size_q', observed=True)['dorm_fee_min'].apply(lambda x: x.isna().mean()*100)
print('\n규모 사분위별 결측률(%):'); print(rate.round(0))

fig, ax = plt.subplots(figsize=(6,4))
ax.bar(range(4), rate.values, color=['#c0392b','#e67e22','#f1c40f','#27ae60'])
ax.set_xticks(range(4)); ax.set_xticklabels(rate.index)
ax.set_ylabel('결측률 (%)'); ax.set_title('Step 4 (MAR): 소규모 대학일수록 기숙사비 미공시')
for i,v in enumerate(rate.values): ax.text(i, v+1, f'{v:.0f}%', ha='center')
plt.tight_layout(); plt.savefig(FIG/'step4_mar_missingness.png', dpi=120); plt.show()
print('\n판정: 결측률이 규모에 단조 감소 → MCAR 아님, MAR. 결측을 삭제/평균대체하면 편향.')

## Step 5 — 이상치 (자연 vs 인공)

모든 이상치를 제거하면 안 됩니다. **자연 이상치**(연세대처럼 실제로 큰 값)는 정보이고, **인공 이상치**(기숙사비 1원 같은 입력 오류/placeholder)는 잡음입니다. 둘을 구분합니다.

In [ ]:
fee = u['dorm_fee_min'].dropna()
q1, q3 = fee.quantile([.25,.75]); iqr = q3-q1
print('기숙사비 min=%.0f max=%.0f (원)' % (fee.min(), fee.max()))
print('IQR 상한 이상치:', int((fee > q3+1.5*iqr).sum()), '개 (대부분 자연 이상치 = 비싼 신축 기숙사)')
art = u.loc[u['dorm_fee_min']<=1, 'univ_name'].tolist()
print('인공 이상치(월 1원 이하, placeholder로 의심):', len(art), art)
print('→ 자연 이상치는 유지, 인공 이상치(≤1원)는 분석에서 결측 처리 권장')

## Step 6 — 변환

우편향(skew 2+)은 로그 변환으로 크게 완화됩니다. 평균·상관 기반 분석의 전제를 맞추기 위함입니다.

In [ ]:
u['log_total'] = np.log1p(u['total'])
print('변환 전 skew: %.2f → 변환 후 skew: %.2f' % (u['total'].skew(), u['log_total'].skew()))
fig, ax = plt.subplots(1,2, figsize=(10,3.2))
ax[0].hist(u['total'], bins=30, color='#95a5a6'); ax[0].set_title('원본 total (skew %.2f)'%u['total'].skew())
ax[1].hist(u['log_total'], bins=30, color='#16a085'); ax[1].set_title('log(1+total) (skew %.2f)'%u['log_total'].skew())
plt.tight_layout(); plt.show()

## Step 7 — 파생변수 ⭐ (핵심 인사이트 2)

원본에 없던 **국적 다양성 지수(Shannon entropy)** 를 만듭니다. 값이 클수록 여러 국적이 고르게 분포, 작을수록 특정 국적 쏠림. 이 변수는 챗봇 답변에 바로 쓰입니다(*"국적이 다양한 대학은?"*, *"특정 국적 의존도가 높은 대학은?"*) → **분석과 제품이 연결**됩니다.

In [ ]:
tmp = s.groupby(['univ_key','nationality']).headcount.sum().reset_index()
def shannon(g):
    p = g / g.sum(); return float(-(p*np.log(p)).sum())
div = tmp.groupby('univ_key').headcount.apply(shannon).rename('diversity')
tot = s.groupby('univ_key').headcount.sum().rename('total_stu')
nat_n = tmp.groupby('univ_key').nationality.nunique().rename('n_nationalities')
# 최대 국적 의존도 = 가장 큰 국적 비중
dep = tmp.groupby('univ_key').headcount.apply(lambda g: g.max()/g.sum()).rename('top_nat_share')
d = pd.concat([div, tot, nat_n, dep], axis=1).join(u.set_index('univ_key')[['univ_name','region']])
big = d[d.total_stu >= 500]
print('국적 다양성 상위5 (유학생 500+):')
print(big.nlargest(5,'diversity')[['univ_name','diversity','n_nationalities','total_stu']].round(2).to_string(index=False))
print('\n국적 쏠림 상위5 (규모 크지만 특정국 의존):')
print(big.nlargest(5,'top_nat_share')[['univ_name','top_nat_share','n_nationalities','total_stu']].round(2).to_string(index=False))

top = big.nlargest(8,'diversity').iloc[::-1]
fig, ax = plt.subplots(figsize=(6,4))
ax.barh(range(len(top)), top.diversity, color='#2980b9')
ax.set_yticks(range(len(top))); ax.set_yticklabels(top.univ_name)
ax.set_xlabel('Shannon 다양성 지수'); ax.set_title('Step 7: 국적 다양성 상위 대학 (유학생 500+)')
plt.tight_layout(); plt.savefig(FIG/'step7_diversity.png', dpi=120); plt.show()

## 결론 — 발표 인사이트 2개

1. **결측은 무작위가 아니다 (Step 4).** 기숙사비 결측률이 규모 사분위별 49%(소) → 5%(대)로 단조 감소. MAR이므로 단순 삭제 시 *기숙사 비용 정보가 대형 대학에 편향*됩니다. 데이터를 채우지 않고 **"왜 비었는가"** 를 신호로 씁니다.
2. **국적 다양성 지수 (Step 7).** 원본에 없던 파생변수. KAIST·서울대는 다양성 3.4+(85~114개국)인 반면, 규모가 큰 일부 대학은 특정 국적 의존도가 높습니다. 이 변수는 챗봇이 *"국적이 다양한 대학"* 질문에 답하는 데 바로 쓰여 **분석 → 제품**으로 이어집니다.

> 발표에서는 이 **두 인사이트만** 슬라이드에 올리고, 7단계 전체는 이 노트북으로 갈음합니다 (SSOT §5-2: 체크리스트 나열 금지).